<a href="https://colab.research.google.com/github/Lornatt88/DriveIQ/blob/main/notebooks/01_explore_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets transformers

from datasets import load_dataset
ds = load_dataset("coastalcph/lex_glue", "unfair_tos")
print(ds)

names = ds["train"].features["labels"].feature.names
print(names)

unfair = sum(1 for x in ds["train"]["labels"] if x)
print(f"Unfair sentences in train: {unfair} / {len(ds['train'])}")

README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

unfair_tos/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  501kB            

unfair_tos/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

unfair_tos/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  147kB            

unfair_tos/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

unfair_tos/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5532 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1607 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2275 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 5532
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 1607
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 2275
    })
})
['Limitation of liability', 'Unilateral termination', 'Unilateral change', 'Content removal', 'Contract by using', 'Choice of law', 'Jurisdiction', 'Arbitration']
Unfair sentences in train: 630 / 5532


In [2]:
from collections import Counter

counts = Counter(label for labs in ds["train"]["labels"] for label in labs)
for i, name in enumerate(names):
    print(f"{name}: {counts[i]}")

multi = sum(1 for x in ds["train"]["labels"] if len(x) > 1)
print(f"Sentences with more than one unfair type: {multi}")

Limitation of liability: 191
Unilateral termination: 139
Unilateral change: 122
Content removal: 73
Contract by using: 76
Choice of law: 39
Jurisdiction: 34
Arbitration: 28
Sentences with more than one unfair type: 63


We can see here that:
- 63 sentences have more than one unfair type, so this is a multi-label problem.
 - The unfair class types are not balanced. With Limitation of liability appearing 191 times and Arbitration a lot less, with only 28 times appearing. This can tell us that the model might struggle with rare types.

In [3]:
for split in ["train", "validation", "test"]:
    labels = ds[split]["labels"]
    fair = sum(1 for x in labels if not x)
    print(f"{split}: {fair}/{len(labels)} fair ({fair/len(labels):.1%})")

train: 4902/5532 fair (88.6%)
validation: 2045/2275 fair (89.9%)
test: 1435/1607 fair (89.3%)


We can see that almost 89% of the sentences are fair, which tells us that the data is very imbalanced and that all three splits have a similar balance.